## Bulk geocoding for multiple addresses

This notebook demonstrates two patterns for bulk or batch-geocoding a number of addresses - based on the type of input data you have - and plot the geocoded results on a map with custom popup. 
1. The first pattern focuses on batch geocoding multiple addresses from a .txt file in your workspace.
2. The second pattern focuses on adding a .csv file in your workspace as an item to the portal and then geocoding from the item.

> Note: If your input address data is in the form of list or dictionaries within the notebook as a cell, you can skip directly to use this [batch_geocode()](https://developers.arcgis.com/python/latest/api-reference/arcgis.geocoding.html#arcgis.geocoding.batch_geocode) method from the ArcGIS API for Python. 


#### Import necessary libraries and connect to GIS

In [1]:
from arcgis.gis import GIS, ItemProperties, ItemTypeEnum
from arcgis.features import FeatureLayer
from arcgis.geocoding import batch_geocode, geocode_from_items
from arcgis.map.popups import PopupInfo, FieldInfo
import pandas as pd

gis = GIS("home")

You are logged on as mmajumdar with an administrator role, proceed with caution.


### Pattern 1: Single line addresses available in a `.txt` file

#### Define variables

In [2]:
txt_file = "<PATH-TO-TXT-FILE-IN-WORKSPACE-WITH-ADDRESSES-TO-GEOCODE>"

#### Access each address in a list

In [3]:
address_list = []
with open(txt_file, 'r') as f:
    address_list = [line.strip() for line in f]
address_list

['2254 MOORE ST, SAN DIEGO CA  92110-3015',
 '9830 SIEMPRE VIVA RD # 17, SAN DIEGO CA  92154-7236',
 '5353 VAN NUYS PL, SAN DIEGO CA  92109-1251',
 '4305 MISSION BLVD, SAN DIEGO CA  92109-3917',
 '4367 ALABAMA ST, SAN DIEGO CA  92104-1023']

#### Geocode this list and save the result as a FeatureSet

In [4]:
result_fs = batch_geocode(addresses = address_list, as_featureset = True)
result_fs

<FeatureSet> 5 features

#### Convert this to a Spatially Enabled DataFrame

In [5]:
result_df = result_fs.df
result_df.head()

,ResultID,Loc_name,Status,Score,Match_addr,LongLabel,ShortLabel,Addr_type,Type,PlaceName,...,Xmin,Xmax,Ymin,Ymax,ExInfo,MatchID,StrucType,StrucDet,OBJECTID,SHAPE
0,2,World,M,100,"5353 Van Nuys Place, San Diego, California, 92109","5353 Van Nuys Place, San Diego, CA, 92109, USA",5353 Van Nuys Place,PointAddress,,,...,-117.254149,-117.252149,32.812118,32.814118,,AQEa9QAAvd8BAJUBBAAAAAAMDlZBTiBOVVlTIFBMQUNFFQ...,,,1,"{""x"": -117.253353370937, ""y"": 32.813071800774,..."
1,3,World,M,100,"4305 Mission Boulevard, San Diego, California,...","4305 Mission Boulevard, San Diego, CA, 92109, USA",4305 Mission Boulevard,PointAddress,,,...,-117.254434,-117.254434,32.793886,32.793886,,AQEZ9QAAqd8BAJUBBAAAAAAMEU1JU1NJT04gQk9VTEVWQV...,,,2,"{""x"": -117.254629515696, ""y"": 32.793828585555,..."
2,0,World,M,100,"2254 Moore Street, San Diego, California, 92110","2254 Moore Street, San Diego, CA, 92110, USA",2254 Moore Street,PointAddress,,,...,-117.192383,-117.192383,32.747923,32.747923,,AQFX9QAAe98BAJUBBAAAAAAMDE1PT1JFIFNUUkVFVBUEMj...,,,3,"{""x"": -117.19249756876, ""y"": 32.747727866633, ..."
3,1,World,M,100,"9830 Siempre Viva Road, Suite 17, San Diego, C...","9830 Siempre Viva Road, Suite 17, San Diego, C...","9830 Siempre Viva Road, Suite 17",Subaddress,,,...,-116.935531,-116.933531,32.556354,32.558354,,AQBZ9gAAvd4BAJUBBAAAAAAMEVNJRU1QUkUgVklWQSBST0...,,,4,"{""x"": -116.934536032814, ""y"": 32.556918953805,..."
4,4,World,M,100,"4367 Alabama St, San Diego, California, 92104","4367 Alabama St, San Diego, CA, 92104, USA",4367 Alabama St,PointAddress,,,...,-117.143436,-117.141436,32.755739,32.757739,,AQGJ9QAAhN8BAJUBBAAAAAAMDkFMQUJBTUEgU1RSRUVUFQ...,,,5,"{""x"": -117.14243623295, ""y"": 32.756739292646, ..."


#### Plot the results on the map

In [6]:
map1 = gis.map()
result_df.spatial.plot(map1)
map1.zoom_to_layer(result_df)
map1

Map(extent={'xmin': -13052725.650841223, 'ymin': 3836638.96644981, 'xmax': -13017093.007320553, 'ymax': 387051…

#### Create a popup and add to the map

In [9]:
popup = map1.content.popup(0)
popup.disable_popup = False
fields = [FieldInfo(field_name=column_name) for column_name in result_df.columns]
popup.edit("Address", field_infos=fields)

True

### Pattern 2: Single line addresses available in a column in a `.csv` file

In [10]:
csv_file = "<PATH-TO-CSV-FILE-IN-WORKSPACE-WITH-ADDRESSES-TO-GEOCODE>"
title = "<TITLE-FOR-CSV-ITEM-TO-CREATE>"

output_title = "<TITLE-FOR-GEOCODED-FEATURE-LAYER-ITEM>"

In [11]:
df = pd.read_csv(csv_file)
df.head()

,ID,Address
0,1,"""676 Howard St, San Francisco, USA"""
1,2,"""TORONTO GENERAL HOSPITAL, TORONTO, CAN"""
2,3,"""Arroyo Galeana Sin Numero, San Pablo Huitzo..."
3,4,"""Binnenhof 13, 6553 CH DEEST"""
4,5,"""Hoge Maasdijk, 4281 NG ANDEL"""


In [7]:
# Define item properties
root_folder = gis.content.folders.get()
item_properties = ItemProperties(title = title, item_type = ItemTypeEnum.CSV)

# Publish as item
job = root_folder.add(item_properties = item_properties, file = csv_file,  stream=True)
item = job.result()
item

<Item title:"address file from Python" type:CSV owner:mmajumdar>

In [10]:
geocoding_result = geocode_from_items(input_data = item, output_type = 'Feature Layer', output_name = output_title,  gis = gis)
geocoding_result

<Item title:"Geocoded_Feature_Service_Geocoded_addresses" type:Feature Layer Collection owner:mmajumdar>

In [13]:
geocoded_df = FeatureLayer(geocoding_result.url+'/0').query(as_df=True)
geocoded_df.head()

,country,ymin,in_postalext,addr_type,in_address3,in_address2,stdir,nbrhd,user_address,displayy,...,addnum,ymax,in_address,x,y,shortlabel,objectid,addbldg,territory,SHAPE
0,CAN,43.654083,,POI,,,,Bay Street Corridor,"""TORONTO GENERAL HOSPITAL, TORONTO, CAN""",43.659083,...,200,43.664083,"""TORONTO GENERAL HOSPITAL, TORONTO, CAN""",-79.387447,43.659373,Toronto General Hospital,1,,,"{""x"": -79.38827699999996, ""y"": 43.659083000000..."
1,NLD,51.785387,,StreetName,,,,,"""Hoge Maasdijk, 4281 NG ANDEL""",51.786387390291,...,,51.787387,"""Hoge Maasdijk, 4281 NG ANDEL""",5.055804,51.786387,Hoge Maasdijk,2,,,"{""x"": 5.055804040000055, ""y"": 51.7863873900000..."
2,USA,37.784612,,PointAddress,,,,,"""676 Howard St, San Francisco, USA""",37.785612013091,...,676,37.786612,"""676 Howard St, San Francisco, USA""",-122.400001,37.78542,676 Howard St,3,,,"{""x"": -122.40015003899998, ""y"": 37.78561201300..."
3,NLD,51.884486,,PointAddress,,,,,"""Binnenhof 13, 6553 CH DEEST""",51.885486171528,...,13,51.886486,"""Binnenhof 13, 6553 CH DEEST""",5.665665,51.885486,Binnenhof 13,4,,,"{""x"": 5.665664722000031, ""y"": 51.8854861720000..."
4,MEX,17.285005,,StreetName,,,,De Guadalupe,"""Arroyo Galeana Sin Numero, San Pablo Huitzo...",17.286004663962,...,,17.287005,"""Arroyo Galeana Sin Numero, San Pablo Huitzo...",-96.883871,17.286005,Privada de Galeana,5,,,"{""x"": -96.88387070899995, ""y"": 17.286004664000..."


#### Publish the results on a map

In [14]:
map2 = gis.map()
map2.content.add(geocoded_df)
map2.zoom_to_layer(geocoded_df)
map2

Map(extent={'xmin': -13625522.375361763, 'ymin': 1954143.2319368646, 'xmax': 630698.9118584573, 'ymax': 677944…

#### Create a popup and add to the map

In [15]:
popup = map2.content.popup(0)
popup.disable_popup = False
fields = [FieldInfo(field_name=column_name) for column_name in geocoded_df.columns]
popup.edit("Address", field_infos=fields)

True